# [Experiment] Parameter-Matched KAN-TabNet | Step LR | Poker Hand

### Overview
This notebook introduces the Kolmogorov-Arnold Network (KAN) architecture into the TabNet framework. The network is specifically configured to match the total trainable parameter count of the original vanilla baseline.

### Methodological Context: Strict Isolation
To precisely isolate the effects of the KAN architecture, we maintain the exact same StepLR optimization environment used in our vanilla control—a schedule chosen to strictly adhere to the original TabNet paper's experimental design. By holding the optimization thermodynamics constant and parameter-matching the models, we guarantee that any performance delta is driven purely by the mathematical flexibility of the B-splines, rather than a brute-force increase in structural capacity or a shifted learning rate.

### The Hypothesis
We hypothesize that substituting standard linear transformations with learnable activation functions on the network edges will fundamentally alter how the model routes and represents features. This notebook serves to evaluate whether this architectural shift improves predictive accuracy or alters convergence behavior when operating under the exact same foundational constraints defined by the original TabNet authors.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url_train = 'https://archive.ics.uci.edu/ml/machine-learning-databases/poker/poker-hand-training-true.data'
url_test = 'https://archive.ics.uci.edu/ml/machine-learning-databases/poker/poker-hand-testing.data'

df_train = pd.read_csv(url_train, header=None)
df_test = pd.read_csv(url_test, header=None)

# 0-index the categorical features for PyTorch embedding layers
X_train_full = df_train.iloc[:, :-1].values - 1
y_train_full = df_train.iloc[:, -1].values.astype(int)

X_test = df_test.iloc[:, :-1].values - 1
y_test = df_test.iloc[:, -1].values.astype(int)

# remove the permutation invariance trap that cripples standard MLPs on this dataset
def sort_poker_hands(X):
    # reshape from (N, 10) to (N, 5, 2) to lock each Suit with its Rank
    X_pairs = X.reshape(-1, 5, 2)

    # get the indices that sort each hand based on the Rank (index 1 of the pair)
    sort_indices = np.argsort(X_pairs[:, :, 1], axis=1)

    # apply the sort to both Suit and Rank simultaneously
    row_indices = np.arange(X.shape[0])[:, np.newaxis]
    X_sorted_pairs = X_pairs[row_indices, sort_indices]

    # flatten back to the standard (N, 10) TabNet input format
    return X_sorted_pairs.reshape(-1, 10)

print("Sorting hands to remove permutation invariance...")
X_train_full = sort_poker_hands(X_train_full)
X_test = sort_poker_hands(X_test)
print("Sorting complete.")

del df_train, df_test
gc.collect()

# split 6k samples for validation from the 25k training dataset
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=6000, random_state=seed, stratify=y_train_full
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Sorting hands to remove permutation invariance...
Sorting complete.
Train shape: (19010, 10)
Valid shape: (6000, 10)
Test shape:  (1000000, 10)


### Model Training

In [5]:
MAX_EPOCHS = 10000

In [6]:
# define categorical structures for the 10 input columns
categorical_indices = list(range(10))
categorical_dimensions = [4, 13, 4, 13, 4, 13, 4, 13, 4, 13]
categorical_embedding_dims = [1] * 10

# Hyperparameters from original paper.
# Notes:
# - momentum is 1 - 0.95 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 100 epochs approximates the paper's 500 iterations
#   (19k train rows / 4096 batch size = ~4.64 iters/epoch. 500 / 4.64 = ~107. Using 100).
paper_config = {
    'n_steps': 4,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.05,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=100, gamma=0.95),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=6, # adjusted so parameter count is matched with vanilla TabNet
    n_a=4, # adjusted so parameter count is matched with vanilla TabNet
    **paper_config,
    cat_idxs=categorical_indices,
    cat_dims=categorical_dimensions,
    cat_emb_dim=categorical_embedding_dims,
    clip_value=2.,
    optimizer_params=dict(lr=0.005), # reduced as 0.01 lr is too aggressive
    use_kan=True,
    kan_grid_size=6,
    kan_spline_order=2,
    seed=seed,
    verbose=250
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 4096,
    'virtual_batch_size': 1024,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=MAX_EPOCHS,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.79905 | valid_accuracy: 0.51083 |  0:00:01s
epoch 250| loss: 0.72208 | valid_accuracy: 0.69483 |  0:02:02s
epoch 500| loss: 0.58593 | valid_accuracy: 0.77667 |  0:04:01s
epoch 750| loss: 0.42456 | valid_accuracy: 0.8625  |  0:06:03s
epoch 1000| loss: 0.23522 | valid_accuracy: 0.92    |  0:08:04s
epoch 1250| loss: 0.13987 | valid_accuracy: 0.94967 |  0:10:05s
epoch 1500| loss: 0.1167  | valid_accuracy: 0.96467 |  0:12:05s
epoch 1750| loss: 0.06647 | valid_accuracy: 0.98067 |  0:14:05s
epoch 2000| loss: 0.04445 | valid_accuracy: 0.98367 |  0:16:06s
epoch 2250| loss: 0.03772 | valid_accuracy: 0.98667 |  0:18:06s
epoch 2500| loss: 0.02881 | valid_accuracy: 0.98783 |  0:20:07s
epoch 2750| loss: 0.02357 | valid_accuracy: 0.988   |  0:22:07s
epoch 3000| loss: 0.02015 | valid_accuracy: 0.989   |  0:24:07s
epoch 3250| loss: 0.01454 | valid_accuracy: 0.9895  |  0:26:07s
epoch 3500| loss: 0.02497 | valid_accuracy: 0.9885  |  0:28:09s
epoch 3750| loss: 0.02316 | valid_accuracy: 

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 25,225


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.99160


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/poker_hand/tables'
MODELS_DIR = f'{BASE_DIR}/results/poker_hand/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Poker Hand",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/02_kan_param_matched_step_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/02_kan_param_matched_step_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/poker_hand/tables/02_kan_param_matched_step_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/poker_hand/models/02_kan_param_matched_step_lr_25k.zip
